In [4]:
import json
import numpy as np
import os
import pickle as pkl
from copy import deepcopy
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import random
import scipy
import scipy.stats
import statsmodels.stats
import statsmodels.stats.weightstats

In [2]:
root = '/home/liuxiao/ContextReasoning/fribble_exp_0shot_logs/multiruns'
model_names = ['vicreg', 'seco', 'dino', 'orl', 'supervised','simclr','simsiam','context_encoder']
model_results_all_rules = []
model_error_bars_all = {k: {} for k in model_names}
model_error_cnt = {k: {} for k in model_names}
for rule_no in [0,2,4]:
    model_results = {}
    for model in model_names:
        if model in ['orl']:
            runs = [0,1,3]
        else:
            runs = [0,2,4]
        model_results[model] = {}
        for run_no in runs:
            with open('{}/{}/{}_{}.pkl'.format(root,model,rule_no,run_no),'rb') as f:
                results_tmp = pkl.load(f)
            
            for entry in results_tmp[0].split('\t'):
                cond, acc1 = entry.split(':')[0], float(entry.split(':')[1].strip())
                cond = cond.replace('normal1','normal')
#                 print(model,rule_no,run_no,cond,acc1)
                model_results[model][cond] =  model_results[model].get(cond, 0) + acc1 * (1/3)
            for pth, pred in results_tmp[1].items():
                gt = {'fa1':0,'fb1':1,'fb3':2,'fc1':3}[pth.split('/')[0].split('_')[1]]
                tmp = pth.split('/')[2].split('_') 
                cond = tmp[-1] if len(tmp) == 7 else 'normal'
                model_error_cnt[model].setdefault(cond,[]).append(1 if gt == pred else 0)

    model_results_all_rules.append(model_results)  
    print(model_results)
for m,_ in model_error_cnt.items():
    for k,v in _.items():
        model_error_bars_all[m][k] = np.std(v)/(len(v)**0.5)

{'vicreg': {'normal': 0.40005034208297724, 'jigsaw1': 0.3062765896320343, 'jigsaw2': 0.26358412702878314, 'jigsaw3': 0.3270508348941803, 'blur1': 0.40984765688578284, 'blur2': 0.38888203104337055, 'blur3': 0.3844927450021108, 'blur4': 0.25650117794672644, 'blur5': 0.21278498073418933, 'amount1': 0.2700340151786804, 'amount2': 0.26134955883026123, 'amount3': 0.36433519919713336, 'amount4': 0.3631538152694702, 'amount5': 0.4219661056995392}, 'seco': {'normal': 0.43577038248380023, 'jigsaw1': 0.3548093239466349, 'jigsaw2': 0.3301205237706502, 'jigsaw3': 0.34987606604894, 'blur1': 0.4215601086616516, 'blur2': 0.4083225826422373, 'blur3': 0.38935961325963336, 'blur4': 0.3030960261821747, 'blur5': 0.2957248389720917, 'amount1': 0.34236909945805866, 'amount2': 0.37231493989626563, 'amount3': 0.3417636652787526, 'amount4': 0.3948477109273274, 'amount5': 0.41399658719698584}, 'dino': {'normal': 0.3473552266756693, 'jigsaw1': 0.30655327439308167, 'jigsaw2': 0.26654939850171405, 'jigsaw3': 0.3363

In [3]:
def bootstrap(population):
#     print(population, int(N)//2)
    sampled = np.random.choice(population , size=int(len(population))//2,replace=False)
    return sum(sampled)/len(sampled)

#### all models, all conditions

In [7]:
# model_error_cnt['vicreg'].keys()

for model, _ in model_error_cnt.items():
    for cond, arr in _.items():
        
        random_list = [0]*int(len(arr)*0.75) + [1]*int(len(arr)*0.25)
        random.shuffle(random_list)
        sampled_list = [bootstrap(arr)  for i in range(10000)]
        chance_list = [bootstrap(random_list)  for i in range(10000)]
        print(model, cond, statsmodels.stats.weightstats.ttest_ind(sampled_list,chance_list))

vicreg jigsaw2 (295.4073366380597, 0.0, 19998.0)
vicreg normal (2088.603231390764, 0.0, 19998.0)
vicreg amount4 (1888.7977024009028, 0.0, 19998.0)
vicreg blur4 (196.60692931637328, 0.0, 19998.0)
vicreg amount2 (1180.670199036866, 0.0, 19998.0)
vicreg amount5 (2256.7260021555876, 0.0, 19998.0)
vicreg amount3 (1736.32088990725, 0.0, 19998.0)
vicreg blur2 (1698.0358766274262, 0.0, 19998.0)
vicreg jigsaw1 (696.6700458726567, 0.0, 19998.0)
vicreg jigsaw3 (889.3555730912545, 0.0, 19998.0)
vicreg blur1 (1662.177290217473, 0.0, 19998.0)
vicreg blur3 (1224.3002415673723, 0.0, 19998.0)
vicreg amount1 (900.9403938162578, 0.0, 19998.0)
vicreg blur5 (-385.57479139671426, 0.0, 19998.0)
seco jigsaw2 (1000.4355066349227, 0.0, 19998.0)
seco normal (2725.0969077530417, 0.0, 19998.0)
seco amount4 (2404.179300476798, 0.0, 19998.0)
seco blur4 (72.85559693429427, 0.0, 19998.0)
seco amount2 (1797.8404813490006, 0.0, 19998.0)
seco amount5 (2245.188977036793, 0.0, 19998.0)
seco amount3 (2170.7874610477643, 0.0

In [5]:
# model_error_cnt['vicreg'].keys()

random_list = [0]*750 + [1]*250
sampled_list = [bootstrap(random_list)  for i in range(10000)]
random.shuffle(random_list)
chance_list = [bootstrap(random_list)  for i in range(10000)]
print(statsmodels.stats.weightstats.ttest_ind(sampled_list,chance_list))

(-0.8947224512114059, 0.3709462422184888, 19998.0)


In [21]:
chance_list

[0.25203252032520324,
 0.2540650406504065,
 0.2577235772357724,
 0.24065040650406505,
 0.25121951219512195,
 0.2483739837398374,
 0.24959349593495936,
 0.2556910569105691,
 0.24878048780487805,
 0.2475609756097561,
 0.25447154471544714,
 0.2483739837398374,
 0.24146341463414633,
 0.25,
 0.2528455284552846,
 0.2630081300813008,
 0.24634146341463414,
 0.2540650406504065,
 0.23780487804878048,
 0.25447154471544714,
 0.2528455284552846,
 0.2483739837398374,
 0.24796747967479674,
 0.25040650406504067,
 0.2540650406504065,
 0.25,
 0.24471544715447155,
 0.25121951219512195,
 0.25934959349593495,
 0.258130081300813,
 0.2508130081300813,
 0.258130081300813,
 0.25853658536585367,
 0.25894308943089434,
 0.25691056910569104,
 0.2483739837398374,
 0.24390243902439024,
 0.2491869918699187,
 0.25894308943089434,
 0.2483739837398374,
 0.25691056910569104,
 0.258130081300813,
 0.2573170731707317,
 0.25,
 0.23780487804878048,
 0.2434959349593496,
 0.25,
 0.25447154471544714,
 0.25894308943089434,
 0.257

In [22]:
sampled_list

[0.2638211382113821,
 0.2613821138211382,
 0.2528455284552846,
 0.2552845528455285,
 0.26097560975609757,
 0.2686991869918699,
 0.27764227642276423,
 0.26951219512195124,
 0.2760162601626016,
 0.27723577235772356,
 0.2678861788617886,
 0.258130081300813,
 0.2658536585365854,
 0.2556910569105691,
 0.2711382113821138,
 0.2597560975609756,
 0.2573170731707317,
 0.2552845528455285,
 0.2646341463414634,
 0.2678861788617886,
 0.2577235772357724,
 0.2613821138211382,
 0.2634146341463415,
 0.2654471544715447,
 0.2686991869918699,
 0.266260162601626,
 0.2646341463414634,
 0.2573170731707317,
 0.2654471544715447,
 0.2605691056910569,
 0.2548780487804878,
 0.2646341463414634,
 0.2731707317073171,
 0.2638211382113821,
 0.2634146341463415,
 0.26260162601626014,
 0.2658536585365854,
 0.25853658536585367,
 0.26178861788617885,
 0.27764227642276423,
 0.26910569105691057,
 0.2678861788617886,
 0.2605691056910569,
 0.25691056910569104,
 0.25853658536585367,
 0.26097560975609757,
 0.26504065040650404,
 0